In [935]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime

# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

# 모든 컬럼 출력설정(선택)
pd.set_option('display.max_columns', None)

#데이터 불러오기 
df = pd.read_csv('total_data.csv',index_col=0)

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")


[행/컬럼 갯수]
행: 51279, 컬럼: 40



In [936]:
#주차 컬럼 날짜타입 변환 (범주->날짜형)
df['주차'] = pd.to_datetime(df['주차'], format='%Y%m%d')
df['주차'].info()

<class 'pandas.core.series.Series'>
Index: 51279 entries, 0 to 51278
Series name: 주차
Non-Null Count  Dtype         
--------------  -----         
51279 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 801.2 KB


In [937]:
# 결측치 확인 -> 없음 
df.isna().sum()

기간        0
주차        0
라인        0
성별        0
기획년도      0
시즌이월      0
상품년차      0
시즌        0
복종        0
소품종       0
CAT       0
총입고수량     0
총입고원가     0
총입고택가     0
총출고수량     0
총출고원가     0
총출고택가     0
판매액       0
판매수량      0
매출원가      0
판매택가      0
총판매액      0
총판매수량     0
총매출원가     0
총판매택가     0
물류재고수량    0
물류재고원가    0
물류재고택가    0
매장재고수량    0
매장재고원가    0
매장재고택가    0
재고수량      0
재고원가      0
재고택가      0
기간입고수량    0
기간입고원가    0
기간입고택가    0
기간출고수량    0
기간출고원가    0
기간출고택가    0
dtype: int64

In [938]:
# 중복값 확인 및 제거 -> 전체 중복 12개
df.duplicated().sum() 
df.drop_duplicates(inplace=True)
# df[df.duplicated(keep=False)].sort_values(by='주차')

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

[행/컬럼 갯수]
행: 51267, 컬럼: 40



# 범주형 컬럼 확인

In [939]:
#성별 소품종 클래스 확인 : 기타로 분류되는 클래스 2개 존재
#--> 최종 '기타' 로 오분류된 항목 52개 변환
display(df['성별'].value_counts())

filtered_df = df[df['성별'].str.contains('기타')]
filtered_df['소품종'].value_counts()

#봄 패딩 베스트만, '기타' 로 분류됨 -> 성별 구분 착오 예상 --> 봄패딩베스트 '1:남성' 값으로 변환 
con = (df['소품종']=='패딩베스트') & (df['시즌']=='봄')
df.loc[con,'성별'] = '1:남성'

df.loc[con,'성별'].value_counts()

성별
1:남성      47346
3:남녀공용     2510
2:여성       1322
4:기타         89
Name: count, dtype: int64

성별
1:남성    52
Name: count, dtype: int64

In [940]:
#시즌이월 컬럼 클래스 확인 : 이월제품 의미 파악 필요
##결론 : 판매예측/할인최적화 모델링시 '이월' 행 삭제 ( -12578 32%) ,년간 매출 집계시 유지
display(df['시즌이월'].value_counts())

#2024년도 기준 시즌/이월 여부 확인 
df_2024 = df[df['기획년도']==2024]
df_2024 = df_2024.drop(columns=['기간','상품년차'])

# # 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df_2024['카테고리'] = df_2024['시즌'] + "_" +  df_2024['복종'] + "_" + df_2024['소품종'] + "_" + df_2024['라인']+ "_" + df_2024['성별']

# #카테고리별 시즌/이월값이 둘다 있는거 소팅 -> '샘플 가을_니트 셔츠_라운드_ZB_1:남성'   확인
df_2024.groupby('카테고리')['시즌이월'].nunique()

# #샘플확인 : '가을_니트 셔츠_라운드_ZB_1:남성' -> 시즌별 마감 이후 이월로 변경 됨 
con = df_2024['카테고리'] == '가을_니트 셔츠_라운드_ZB_1:남성'
smpl = df_2024[con].sort_values(by='주차',ascending=True)
smpl[(smpl['주차'] >'2024-11-01') & (smpl['주차'] <='2024-12-30')]

# 시즌-> 이월 바뀌는 시점 함수화 : gpt
def find_transition_points(group):
    group = group.sort_values('주차')
    transition_rows = group[(group['시즌이월'].shift(1) == '01_시즌') & (group['시즌이월'] == '02_이월')]
    return transition_rows[['카테고리', '주차']]

transition_points = df_2024.groupby('카테고리', group_keys=False).apply(find_transition_points)

# 시즌 -> 이월로 바뀌는 주차만 출력
transition_points['주차'].unique()

시즌이월
01_시즌    38689
02_이월    12578
Name: count, dtype: int64

<DatetimeArray>
['2024-12-01 00:00:00', '2024-06-02 00:00:00', '2024-10-06 00:00:00']
Length: 3, dtype: datetime64[ns]

# 수치형 컬럼 & 집계 컬럼 점검
사용 컬럼 : '총입고수량','총입고원가','총입고택가','총출고수량','총출고원가','총출고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가'

In [941]:
# 파생변수 생성
#1. 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df['카테고리'] = df['시즌'] + "_" +  df['복종'] + "_" + df['소품종'] + "_" + df['라인']
num_df = df[['카테고리','주차','총입고수량','총입고원가','총입고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가','시즌','복종','소품종','라인','시즌이월']]

#2. 입고기준 원가/택가 
num_df['제품원가'] = (num_df['총입고원가'] / num_df['총입고수량'])
num_df['제품택가'] = (num_df['총입고택가'] / num_df['총입고수량'])

df['카테고리'].nunique()

252

In [942]:
# 총입고수량/입고원가/입고택가  -> 전처리 (-162행)
# 입고전 데이터 확인 및 행 삭제 : 162개 -> 입고되지 않은 상품은 출고 및 판매 불가, 예약판매 등 특수한 케이스 없다고 가정 

print((num_df['총입고수량'] == 0).sum())
filtered_df = num_df[(num_df['총입고수량'] > 0)]

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

162
[행/컬럼 갯수]
행: 51105, 컬럼: 23



In [943]:
# 재고수량/재고원가/재고택가 정합성 확인 -> 재고 관련 컬럼 삭제 
# 결론: 재고관련 집계 컬럼 삭제 후 입고-판매 기준 다시 집계 (입고,판매 데이터의 신뢰도가 더 높다고 봄, 실제 wms랑 비교 할 수 없으므로 가정)
filtered_df['재고잔량_check'] = (filtered_df['총입고수량'] - filtered_df['총판매수량'] == filtered_df['재고수량'])
filtered_df['재고원가_check'] = (filtered_df['총입고원가'] - filtered_df['총매출원가'] == filtered_df['재고원가'])
filtered_df['재고택가_check'] = (filtered_df['총입고택가'] - filtered_df['총판매택가'] == filtered_df['재고원가'])

# 입고 - 판매 = 재고 안맞는 행 : 27569행
filtered_df[filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False]
(filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False).sum() #27569행

# 재고관련 컬럼 삭제
filtered_df.drop(columns=['재고수량','재고원가','재고택가','재고잔량_check','재고원가_check','재고택가_check'],inplace=True)

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

[행/컬럼 갯수]
행: 51105, 컬럼: 20



# 판매수량/판매액/매출원가/판매택가 확인


In [944]:
# 음수 데이터 확인 -> 판매수량/판매액의 음수값 갯수가 다 다름
minus_con = filtered_df.select_dtypes(include='number') <0
minus_con.sum()

총입고수량       0
총입고원가       0
총입고택가       0
판매수량     1795
판매액      1508
매출원가     1863
판매택가     1795
총판매액        2
총판매수량       3
총매출원가       3
총판매택가       3
제품원가        0
제품택가        0
dtype: int64

In [945]:
# 판매데이터 정합성 확인 : 개당 입고택가 와 개당 판매택가 일치 여부 확인 (원가는 입고/판매 원가 다르므로 생략 (제품 클레임 등))
#결론 : 판매택가 정합성 ok 
con = (filtered_df['판매수량'] * filtered_df['제품택가'] != filtered_df['판매택가'])
filtered_df[con]

,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,제품원가,제품택가


In [946]:
# 1. 판매수량은 0인데, 판매액/매출원가가 있는 경우 -> 348행 대치 완료 
#결론: 판매 없이 판매액/매출원가/판매택가 발생할수없다고 판단, 이상치로 간주하고 매출원가/판매택가 0으로 변경
con1 = filtered_df['판매수량'] == 0
con2 = filtered_df['판매액'] != 0
con3 = filtered_df['매출원가'] != 0

print((con1 & (con2 | con3)).sum())

#수량이 0일떄, 판매액/매출원가 0으로 변경
filtered_df.loc[(con1 & (con2 | con3)),['판매수량','판매액','매출원가']]=0

filtered_df[con1].describe()

348


,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,제품원가,제품택가
count,7818,7818.000000,7.818000e+03,7.818000e+03,7818.0,7818.0,7818.0,7818.0,7.818000e+03,7818.000000,7.818000e+03,7.818000e+03,7818.000000,7.818000e+03
mean,2023-04-12 22:45:57.329240064,5090.849706,6.139490e+07,5.447658e+08,0.0,0.0,0.0,0.0,1.000217e+08,2900.524942,3.584569e+07,3.112481e+08,22411.038014,1.532622e+05
min,2021-01-03 00:00:00,1.000000,4.391500e+04,1.600000e+05,0.0,0.0,0.0,0.0,-9.900000e+04,-1.000000,-1.493000e+04,-9.900000e+04,166.000000,3.300000e+03
25%,2022-04-10 00:00:00,1000.000000,1.585521e+07,1.297400e+08,0.0,0.0,0.0,0.0,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,8214.479853,7.990000e+04
50%,2023-05-14 00:00:00,2476.000000,3.742508e+07,3.012170e+08,0.0,0.0,0.0,0.0,5.803200e+07,1198.000000,1.942430e+07,1.597523e+08,13593.708539,9.990000e+04
75%,2024-03-10 00:00:00,4992.000000,7.627107e+07,6.983010e+08,0.0,0.0,0.0,0.0,1.230972e+08,3017.000000,4.443482e+07,3.972910e+08,26914.839740,1.990000e+05
max,2024-12-29 00:00:00,182459.000000,7.369345e+08,9.104704e+09,0.0,0.0,0.0,0.0,2.023863e+09,106352.000000,5.749144e+08,5.306965e+09,277921.000000,1.299000e+06
std,NaN,8979.126126,7.666198e+07,6.934383e+08,0.0,0.0,0.0,0.0,1.636087e+08,6396.793839,6.038977e+07,5.022997e+08,26432.083484,1.255064e+05


In [947]:
# 2-1. 판매수량 >0 , 집계컬럼 <=0 (매출원가)
# 결론 : 4행 존재 집계 오류 판단 -> 입고원가 * 판매수량 으로 대치
con = (filtered_df['판매수량'] > 0) & (filtered_df['매출원가'] < 0)
display(con.sum())

filtered_df.loc[con,'매출원가']= filtered_df['제품원가'].round(0) * filtered_df['판매수량'] 
filtered_df[con]

# 2-2. 판매수량 <0 , 집계컬럼 >=0 (매출원가)
# 결론 : 3행 존재 집계 오류 판단 -> 입고원가 * 판매수량 으로 대치
con = (filtered_df['판매수량'] < 0) & (filtered_df['매출원가'] >= 0)

filtered_df.loc[con,'매출원가']= filtered_df['제품원가'].round(0) * filtered_df['판매수량'] 
filtered_df[con]

4

,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,제품원가,제품택가
11279,여름_니트 셔츠_라운드_ZB,2021-12-19,95434,289446297,2767586000,-1,0,-3033,-29000,452178679,48430,146928382,1404470000,여름,니트 셔츠,라운드,ZB,02_이월,3032.947346,29000.0
15469,봄_코트_싱글코트_ZB,2022-06-12,4485,135692109,1475565000,-1,-59000,-30255,-329000,346254799,2336,73110572,768544000,봄,코트,싱글코트,ZB,02_이월,30254.650836,329000.0
22999,여름_팬츠_니트팬츠_ZE,2022-12-04,10152,52045783,709624800,-1,-21920,-5127,-69900,65380713,4024,20810904,281277600,여름,팬츠,니트팬츠,ZE,02_이월,5126.653172,69900.0


# 여기 부터 평균 실판가 생성!!!!!!!

In [948]:
# 카테고리별 평균 판매수량, 평균 실판가 집계 (아래 고려사항)
# 1. 카테고리별 평균 판매수량(판매수량이 양수값일때의)
# 2. 카테고리별 평균 실판가(판매액/판매수량이 양수값 일때의)
# 3. 시즌 마감일자 기간 까지의 평균 값 적용 

print('[최초 행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

#시즌별 마감일자 확인 (봄 5월말/여름 9월말/가을 11월말 )
filtered_df['월'] = filtered_df['주차'].dt.month

sp_con = (filtered_df['시즌']=='봄') & (filtered_df['월'] >= 6)
su_con = (filtered_df['시즌']=='여름') & (filtered_df['월'] >= 10)
fa_con = (filtered_df['시즌']=='가을') & (filtered_df['월'] >= 12)
avg_df = filtered_df[~(sp_con | su_con | fa_con)]

print('[마감일자까지 행/컬럼 갯수]')
print(f"행: {avg_df.shape[0]}, 컬럼: {avg_df.shape[1]}\n")

#양수값만 확인
plus_con = (avg_df['판매수량'] > 0) & (avg_df['판매액'] > 0)
avg_df = avg_df[plus_con]

print('[양수만 행/컬럼 갯수]')
print(f"행: {avg_df.shape[0]}, 컬럼: {avg_df.shape[1]}\n")

# 평균 판매수량 / 실판가 값 생성 (평균 실판가 = 판매액/판매수량 의 평균 )
# avg_df.describe(include='all')
avg_df['평균실판가'] = avg_df['판매액'] / avg_df['판매수량']

avg_df2 =avg_df.groupby('카테고리')[['판매수량','평균실판가']].mean().reset_index()

[최초 행/컬럼 갯수]
행: 51105, 컬럼: 20

[마감일자까지 행/컬럼 갯수]
행: 38527, 컬럼: 21

[양수만 행/컬럼 갯수]
행: 33750, 컬럼: 21



In [949]:
# 각 데이터프레임의 카테고리 개수 확인
filtered_count = filtered_df['카테고리'].nunique()
avg_count = avg_df2['카테고리'].nunique()

print(f"최초 카테고리 개수: {filtered_count}")
print(f"avg 카테고리 개수: {avg_count}")

#전체 merge
merge_df = filtered_df.merge(avg_df2, on='카테고리', how='inner')
merge_df = merge_df.rename(columns={'판매수량_x' : '판매수량','판매수량_y' : '평균판매수량'})
merge_df.head(3)

최초 카테고리 개수: 252
avg 카테고리 개수: 252


,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,제품원가,제품택가,월,평균판매수량,평균실판가
0,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,0,0,0,84850,3,26560,209700,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,8853.422341,69900.0,1,108.023256,30323.420923
1,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,9029,73056869,631127100,89,5009759,720259,6221100,10134633,211,1707488,14748900,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,8091.357736,69900.0,1,108.023256,30323.420923
2,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2995,21348695,209350500,39,2547480,277992,2726100,3316380,50,356406,3495000,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,7128.111853,69900.0,1,108.023256,30323.420923


In [950]:
# 3. 판매수량 & 판매액 컬럼 이상치 -> 판매수량기준 * 평균 실판가로 대치 (총 658행)

#판매수량 >0 , 집계컬럼 <0 (판매액)
#총 163행 
con1 = (merge_df['판매수량'] > 0) & (merge_df['판매액'] <= 0)
display(con1.sum())

#판매수량 <0 , 집계컬럼 >0 (판매액)
#총 495행 
con2 = (merge_df['판매수량'] < 0) & (merge_df['판매액'] >= 0)
display(con2.sum())

#판매수량 !=0 , 집계컬럼 ==0 (판매액) 
#총 284행 
con3 = (merge_df['판매수량'] != 0) & (merge_df['판매액'] == 0)
display(con3.sum())

#총 이상치 갯수 : 658행
(con1 | con2 | con3).sum()

merge_df.loc[(con1 | con2 | con3),'판매액'] = merge_df['판매수량'] *  merge_df['평균실판가'] 

merge_df.drop(columns=['월','총판매액','총판매수량','총판매택가','총매출원가'],inplace=True)

163

495

284

In [951]:
# 전체 EDA 용 파일 :merge_df 
print('[행/컬럼 갯수]')
print(f"행: {merge_df.shape[0]}, 컬럼: {merge_df.shape[1]}\n")

#컬럼정리 : 컬럼정렬 
final_df = merge_df[['시즌','복종','소품종','라인','시즌이월','카테고리','주차','총입고수량','총입고원가','총입고택가','판매수량','판매액',	'매출원가','판매택가','제품택가','제품원가','평균판매수량','평균실판가']]
final_df.head(3)

[행/컬럼 갯수]
행: 51105, 컬럼: 18



,시즌,복종,소품종,라인,시즌이월,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품택가,제품원가,평균판매수량,평균실판가
0,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,0.0,0,0,69900.0,8853.422341,108.023256,30323.420923
1,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,9029,73056869,631127100,89,5009759.0,720259,6221100,69900.0,8091.357736,108.023256,30323.420923
2,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2995,21348695,209350500,39,2547480.0,277992,2726100,69900.0,7128.111853,108.023256,30323.420923


In [ ]:
#파생변수 생성 : 제품실판가,할인율
final_df['제품실판가'] = final_df['판매액']/final_df['판매수량'] #null 값 있음
final_df['할인율'] = (final_df['판매택가'] - final_df['판매액']) / final_df['판매택가']*100 #null 값 있음

#실수형 서식 변환 : 반올림0까지
round_cols = ['판매액', '제품택가', '제품원가', '평균판매수량', '평균실판가', '제품실판가', '할인율']
final_df[round_cols] = final_df[round_cols].round(0)

# final_df.to_excel('EDA_data.xlsx')

# 최종 eda용 전처리 파일 : final_df

In [953]:
#이상치 판매수량 - 판매액 관련 고찰 

#추정 실판가(판매액/판매수량) 과 직전 실판가 비교  / 추정 판매수량 과 직전 판매수량 비교
# con1 = (filtered_df['판매수량'] > 0) & (filtered_df['판매액'] < 0)
# con2 = (filtered_df['판매수량'] < 0) & (filtered_df['판매액'] > 0)
# display((con1 | con2).sum())

# # 1. 데이터 정렬 (카테고리, 총입고수량, 주차 기준)
# filtered_df = filtered_df.sort_values(by=['카테고리','총입고수량','주차'])

# # 2. 직전 주차의 실판가 계산 & 직전 주차 판매수량
# filtered_df['직전주차_판매수량'] = filtered_df.groupby(['카테고리','총입고수량'])['판매수량'].shift(1)
# filtered_df['직전주차_판매액'] = filtered_df.groupby(['카테고리','총입고수량'])['판매액'].shift(1)

# filtered_df['직전주차_실판가'] = (filtered_df['직전주차_판매액'] / filtered_df['직전주차_판매수량']).round(0)
# filtered_df['추정_실판가'] = (filtered_df['판매액']/ filtered_df['판매수량']).round(0)
# # filtered_df['추정_실판가'] = filtered_df['추정_실판가'].fillna(0)  # NaN 값 0으로 대체
# filtered_df.loc[(con1|con2),['카테고리','주차','판매수량','판매액','직전주차_판매수량','추정_실판가','직전주차_실판가','매출원가','판매택가']]

# # 엑셀로 점검 
# a = filtered_df.loc[(con1|con2),['카테고리','주차','판매수량','판매액','직전주차_판매수량','추정_실판가','직전주차_실판가','매출원가','판매택가']]
# a.to_excel('check_sales.xlsx')

#판매액오류 가능성이 더 커보임 => 최종 : 해당 카테고리별 양수값의 평균 실판가로 대치 시키자 
## 1. 직전주차 실판가 로 대치시킨다면, 0이하이거나 null인 행 57개 추가로 어떻게 대치 시킬지 

# 시계열 , 할인 모델링 전처리 진행
-라인 제거/ 소품종 제거
-시즌이월 '이월' 제거
-카테고리별 수치형 그룹화 /파생변수 생성

In [954]:
# 특정 라인(ZD, ZE, ZF) 제거
final_df = final_df[~final_df['라인'].isin(['ZD', 'ZE', 'ZF'])]

# '소품', '언더웨어' 제거
final_df = final_df[~final_df['복종'].isin(['소품', '언더웨어'])]

# 시즌이월 '이월' 제거
final_df = final_df[~final_df['시즌이월'].isin(['02_이월'])]

print('[행/컬럼 갯수]')
print(f"행: {final_df.shape[0]}, 컬럼: {final_df.shape[1]}\n")
final_df.head(5)

[행/컬럼 갯수]
행: 24074, 컬럼: 20



,시즌,복종,소품종,라인,시즌이월,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품택가,제품원가,평균판매수량,평균실판가,제품실판가,할인율
0,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,0.0,0,0,69900.0,8853.0,108.0,30323.0,NaN,NaN
1,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,9029,73056869,631127100,89,5009759.0,720259,6221100,69900.0,8091.0,108.0,30323.0,56289.0,19.0
2,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2995,21348695,209350500,39,2547480.0,277992,2726100,69900.0,7128.0,108.0,30323.0,65320.0,7.0
3,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-10,2247,19893640,157065300,5,188800.0,44267,349500,69900.0,8853.0,108.0,30323.0,37760.0,46.0
4,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-10,9183,74302262,641891700,159,5447170.0,1286538,11114100,69900.0,8091.0,108.0,30323.0,34259.0,51.0


In [955]:
# 그룹화 및 집계 연산 적용 :최종 컬럼 8867행
group_cols = ['시즌','복종','소품종','라인','카테고리','주차']
agg_dict = {
    '총입고수량': 'sum',
    '총입고원가': 'sum',
    '총입고택가': 'sum',

    '판매수량': 'sum',
    '판매액': 'sum',
    '매출원가': 'sum',
    '판매택가': 'sum',
    
    '제품원가' : 'mean',
    '제품택가' : 'mean',
    '제품실판가' : 'mean',
    '할인율' : 'mean',

    '평균판매수량' : 'mean',
    '평균실판가' : 'mean',
    
}
final_df = final_df.groupby(group_cols).agg(agg_dict).reset_index()

In [956]:
#파생변수 : 누적판매율(수량),ROI(수익율), 누적판매액,누적판매수량,누적매출원가,누적판매택가

# 1. 누적판매데이터: cumsum()
final_df = final_df.sort_values(by=['카테고리','주차'])

final_df['누적판매수량'] = final_df.groupby(['카테고리','주차'])['판매수량'].cumsum()
final_df['누적판매액'] = final_df.groupby(['카테고리','주차'])['판매액'].cumsum()
final_df['누적매출원가'] = final_df.groupby(['카테고리','주차'])['매출원가'].cumsum()
final_df['누적판매택가'] = final_df.groupby(['카테고리','주차'])['판매택가'].cumsum()

# 2. 판매율 / roi
final_df['누적판매율'] = final_df['누적판매수량']/final_df['총입고원가']*100
final_df['ROI'] = (final_df['누적판매액']/1.1 - final_df['누적매출원가'])/final_df['총입고원가']

preprocessing_df = final_df.copy()
# final_df.to_csv('tableau_check.csv')

# 모델링 최종 전처리 df : preprocessing_df 
# 문제!!!!!!!! 이상치 데이터가 제대로 제거가 안된듯 코드 점검 요망!!!!

In [957]:
final_df[(final_df['판매액']<0) & (final_df['판매수량']>=0)]

,시즌,복종,소품종,라인,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,제품실판가,할인율,평균판매수량,평균실판가,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI
1084,겨울,사파리,패딩사파리,ZB,겨울_사파리_패딩사파리_ZB,2023-10-08,10000,340916863,3990000000,0,-118101.0,1733,0,34212.000000,399000.000000,285663.000000,28.500000,348.0,187484.0,0,-118101.0,1733,0,0.000000e+00,-0.000320
1088,겨울,사파리,패딩사파리,ZB,겨울_사파리_패딩사파리_ZB,2023-11-05,37513,1888438979,16973887000,14,-3282600.0,-879008,1086000,56842.500000,474000.000000,299628.500000,38.750000,348.0,187484.0,14,-3282600.0,-879008,1086000,7.413531e-07,-0.001115
2183,겨울,자켓,싱글재킷,ZA,겨울_자켓_싱글재킷_ZA,2024-12-29,10296,431502948,3379704000,16,-4177761.0,-1666939,-8716000,42771.666667,332333.333333,164285.000000,50.666667,129.0,195500.0,16,-4177761.0,-1666939,-8716000,3.707970e-06,-0.004939
2316,겨울,점퍼,패딩점퍼,ZB,겨울_점퍼_패딩점퍼_ZB,2024-09-15,8030,254473597,1747970000,0,-15100.0,-4792,-50000,32296.000000,224000.000000,166750.000000,25.000000,237.0,118491.0,0,-15100.0,-4792,-50000,0.000000e+00,-0.000035
2917,봄,니트 셔츠,라운드,ZB,봄_니트 셔츠_라운드_ZB,2024-05-12,8906,59523700,763614000,8,-184500.0,45551,372000,6766.000000,89000.000000,44988.000000,50.500000,484.0,31245.0,8,-184500.0,45551,372000,1.344002e-05,-0.003583
4500,사계절,데님,데님팬츠,ZB,사계절_데님_데님팬츠_ZB,2022-08-21,27469,362042300,2744153100,59,-439720.0,697614,5894100,12950.125000,99900.000000,71836.500000,28.125000,70.0,52071.0,59,-439720.0,697614,5894100,1.629644e-05,-0.003031
4563,사계절,데님,데님팬츠,ZB,사계절_데님_데님팬츠_ZB,2023-11-19,19971,352797117,2055102900,14,-790750.0,145878,378600,17671.800000,101900.000000,64684.200000,36.800000,70.0,52071.0,14,-790750.0,145878,378600,3.968286e-06,-0.002451
5819,사계절,팬츠,팬츠(일반),ZB,사계절_팬츠_팬츠(일반)_ZB,2021-07-25,55955,602649386,4203184500,35,-6316759.0,9642,966500,11163.750000,77400.000000,46450.750000,40.250000,324.0,48172.0,35,-6316759.0,9642,966500,5.807689e-06,-0.009545
5872,사계절,팬츠,팬츠(일반),ZB,사계절_팬츠_팬츠(일반)_ZB,2022-07-31,83853,978582448,8376914700,23,-250306.0,95041,2297700,12047.250000,99900.000000,48880.333333,51.000000,324.0,48172.0,23,-250306.0,95041,2297700,2.350338e-06,-0.000330
6425,여름,수트,블레이져(수트),ZB,여름_수트_블레이져(수트)_ZB,2021-09-12,10440,431233322,2077560000,0,-80624.0,-17146,0,41995.000000,199000.000000,156105.600000,21.600000,68.0,177608.0,0,-80624.0,-17146,0,0.000000e+00,-0.000130


In [958]:
# 판매수량 - 매출원가/판매택가 확인 
minus_con = preprocessing_df.select_dtypes(include='number') <0
minus_con.sum()

총입고수량       0
총입고원가       0
총입고택가       0
판매수량      121
판매액       136
매출원가      129
판매택가      124
제품원가        0
제품택가        0
제품실판가       0
할인율        47
평균판매수량      0
평균실판가       0
누적판매수량    121
누적판매액     136
누적매출원가    129
누적판매택가    124
누적판매율     121
ROI       144
dtype: int64

# 판매수량 음수값 기준 이상치 확인
- 1. 1.5 iqr
- 2. 3시그마
- 3. 매장기준 -200 절대값


In [959]:
# 판매수량이 음수인 값만 필터링
negative_sales_df = preprocessing_df[preprocessing_df['판매수량'] < 0]

# IQR 계산
Q1 = negative_sales_df['판매수량'].quantile(0.25)
Q3 = negative_sales_df['판매수량'].quantile(0.75)
IQR = Q3 - Q1

# 3 * IQR 기준으로 이상치 판별
lower_bound = Q1 - 1.5 * IQR

# 이상치 개수 확인
outliers_count = (negative_sales_df['판매수량'] < lower_bound).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound}, 3 IQR 이상인 이상치 개수: {outliers_count}")

이상치 기점: -246.0, 3 IQR 이상인 이상치 개수: 20


In [960]:
# 3표준편차 -> 하한 지점이 너무 낮음 -> 탈락 
mean = negative_sales_df['판매수량'].mean()
stddev = negative_sales_df['판매수량'].std()
lower_bound2 = mean - 3 * stddev  # 3표준편차 하한

# 이상치 개수 확인
outliers_count2 = (negative_sales_df['판매수량'] < lower_bound2).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound2}, 3시그마 이상인 이상치 개수: {outliers_count2}")

이상치 기점: -1157.0053274530037, 3시그마 이상인 이상치 개수: 4


In [961]:
# 전국 매장 수 기준 : 약 200개
# 이상치 개수 확인
outliers_count3 = (negative_sales_df['판매수량'] < -200).sum()

# 결과 출력
print(f"이상치 기점: -200, 3 IQR 이상인 이상치 개수: {outliers_count3}")

이상치 기점: -200, 3 IQR 이상인 이상치 개수: 21


In [962]:
# 카테고리별 개수 세기 - 1.5 iqr 기준
category_counts = negative_sales_df[negative_sales_df['판매수량'] < lower_bound]
category_outlier_counts = category_counts['카테고리'].value_counts()
category_outlier_counts

카테고리
사계절_니트 셔츠_라운드_ZB      7
사계절_데님_데님팬츠_ZB        6
여름_니트 셔츠_라운드_ZB       2
여름_자켓_싱글재킷_ZB         2
겨울_스웨터_라운드_ZB         1
봄_자켓_싱글재킷_ZB          1
사계절_우븐 셔츠_드레스셔츠_ZB    1
Name: count, dtype: int64

# 이상치 가 매출 어디서 / 얼마나 뺄것인가?
#1. 종한 say 
- 평균 수량 : 80
- 현 주차 판매수량: -200
- 가 매출주차 수량 : 500 이라면 ,-(200 + 80) => 220 

#입고수량 유지하면서 현주차 판매수량 평균으로 대체 : 500+(-200) = 220+80

- 전처리 (실판가 & 판매액) : 이상치들 -> 카테고리별 평균 실판가 
- 파생변수 : 판매액 / 매출원가/판매택가 / 할인율 / 총~(누적 판매 데이터) /누적 판매율 / 재고데이터
